In [ ]:
import numpy as np
import os
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, GlobalAveragePooling1D,
    Dense, Dropout, BatchNormalization
)
import matplotlib.pyplot as plt



In [ ]:
def window(signal, fs=173.61, window_sec=5.0, step_sec=1.0):
    window_size = int(fs * window_sec)   
    step_size = int(fs * step_sec)       

    windows = []
    for start in range(0, len(signal) - window_size + 1, step_size):
        windows.append(signal[start:start + window_size])

    return windows

In [ ]:

def load_dataset():
    X = []
    y = []

    class_map = {
        "A": 0,  # Normal
        "C": 1,  # Interictal
        "E": 2   # Ictal
    }

    for folder, label in class_map.items():
        folder_path = os.path.join("./data/Training", folder)

        for file in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file)

            signal = np.loadtxt(file_path)
            windows = window(signal)     

            for w in windows:
                X.append(w)
                y.append(label)

    X = np.array(X)
    y = np.array(y)

    return X, y


In [ ]:
# Load dataset once
X, y = load_dataset()


In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
# Reshape for CNN (before splitting)
X = X.reshape(X.shape[0], X.shape[1], 1)
print(X.shape)

In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train, num_classes=3)
y_val = to_categorical(y_val, num_classes=3)
y_test = to_categorical(y_test, num_classes=3)

print(f"Training set: X{X_train.shape}, y{y_train.shape}")
print(f"Validation set: X{X_val.shape}, y{y_val.shape}")
print(f"Test set: X{X_test.shape}, y{y_test.shape}")

In [ ]:

def build():
    model = Sequential([

        Conv1D(
            filters=4,
            kernel_size=6,
            strides=1,
            activation="relu",
            input_shape=(868, 1)
        ),

        MaxPooling1D(pool_size=2, strides=2),

        Conv1D(
            filters=4,
            kernel_size=5,
            strides=1,
            activation="relu"
        ),

        MaxPooling1D(pool_size=2, strides=2),

        Conv1D(
            filters=10,
            kernel_size=4,
            strides=1,
            activation="relu"
        ),

        MaxPooling1D(pool_size=2, strides=2),

        Conv1D(
            filters=10,
            kernel_size=4,
            strides=1,
            activation="relu"
        ),

        MaxPooling1D(pool_size=2, strides=2),

        Conv1D(
            filters=15,
            kernel_size=4,
            strides=1,
            activation="relu"
        ),

        MaxPooling1D(pool_size=2, strides=2),

        GlobalAveragePooling1D(),

        Dense(50, activation="relu"),

        Dense(20, activation="relu"),

        Dense(3, activation="softmax")
    ])

    return model


model = build()
model.summary()



In [ ]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ],
    shuffle=True
)

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Accuracy: {test_accuracy*100:.4f}")
final_train_accuracy = history.history["accuracy"][-1]
print(f"Final Training Accuracy: {final_train_accuracy*100:.4f}")


In [ ]:

train_loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs = range(1, len(train_loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_loss, label="Training Loss")
plt.plot(epochs, val_loss, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss vs Epoch")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:

y_pred = model.predict(X_test)  
y_pred_classes = np.argmax(y_pred, axis=1)  
y_true_classes = np.argmax(y_test, axis=1)

plt.figure(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true_classes,
    y_pred_classes,
    display_labels=["Normal", "Interictal", "Ictal"],
    cmap="Reds",
    values_format="d"
)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()